# EDA Part 1: Dataset Overview

**Author:** Shreya Aggarwal  
**Date:** 2026-02-22  
**GitHub:** https://github.com/shreya1297/Thesis

---

## What is this notebook?

This notebook gives a **bird's eye view** of the dataset before looking at any images.

We answer:
- How many samples do we have?
- How are they split by condition (Quiescent vs TNF) and protein (RhoA vs RhoB)?
- How long are the recordings (number of time frames)?
- Are the samples spread across multiple experiment dates, or all from one batch?

## Why does this matter?

Before doing any analysis, you need to know the shape of your data. If one condition has only 1 sample and another has 18, any comparison must account for that imbalance. If all samples come from the same day, results might reflect that day's equipment calibration rather than biology.

## Connection to Research Questions

- **SQ3** asks us to compare Quiescent vs TNF cells statistically. But RhoA+TNF has only **n=1** — too few for statistics. This section makes that imbalance visible upfront.
- **Main RQ** requires time-lapse data → we need enough frames per sample to track cells. This section checks recording lengths.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# ── Find project root ─────────────────────────────────────────────────────
# Walk upward from current directory to find the project root.
# This works no matter where you launched Jupyter from.
_p = Path().resolve()
PROJECT_ROOT = None
for _parent in [_p] + list(_p.parents):
    if (_parent / 'Quantification_EndothelialCells_CPM').is_dir():
        PROJECT_ROOT = _parent
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Cannot find project root. '
        'Run Jupyter from inside the implementation/ folder.'
    )

THESIS_DIR   = PROJECT_ROOT / 'Quantification_EndothelialCells_CPM'
OUTPUT_DIR   = THESIS_DIR / 'output'
EDA_DIR      = OUTPUT_DIR / 'eda' / 'eda_01'
EDA_DIR.mkdir(parents=True, exist_ok=True)

# Path to the segmentation batch summary we'll use as our dataset table
SUMMARY_CSV  = OUTPUT_DIR / 'batch_t0001_cpsam_tiff' / 'batch_summary.csv'

print(f'Project root : {PROJECT_ROOT}')
print(f'EDA figures  : {EDA_DIR}')
print(f'Summary CSV  : {SUMMARY_CSV}')

## 1. Loading the Dataset Inventory

We already ran Cellpose segmentation on the first frame (T0001) of all 23 samples.
The results were saved in a CSV file — one row per sample.

We load that CSV here. It also tells us the protein, condition, and number of frames
for every sample, so it doubles as our **dataset inventory**.

In [ ]:
df = pd.read_csv(SUMMARY_CSV)

print(f'Rows (samples): {len(df)}')
print(f'Columns       : {list(df.columns)}')
df.head(3)

## 2. Corpus Overview — the big picture

First, let's count the basics:
- Total samples
- Total frames (time points) across all samples
- How many Q vs TNF, RhoA vs RhoB

In [ ]:
print('='*50)
print('DATASET SUMMARY')
print('='*50)
print(f'Total samples : {len(df)}')
print(f'Total frames  : {df["n_frames"].sum():,}')
print(f'Frames range  : {df["n_frames"].min()} – {df["n_frames"].max()} per sample')
print()
print('By condition:')
print(df['condition'].value_counts().to_string())
print()
print('By protein:')
print(df['protein'].value_counts().to_string())
print()
print('By experiment folder:')
print(df['folder'].value_counts().to_string())

## 3. Condition × Protein Breakdown

Our two main groupings are:
- **Condition**: Quiescent (Q) vs TNF — this is the biological comparison we care about most
- **Protein**: RhoA vs RhoB — these are the two variants of Rho GTPase being studied

This gives us a 2×2 table (four possible combinations).
The number in each cell tells us how many samples we have for that group.

> **Why does this matter?** For statistical testing later (SQ3), we need enough
> samples per group. Groups with n=1 can only be described, not tested.
> Groups with n≥3 are the minimum for a meaningful comparison.

In [ ]:
# Cross-tabulation: protein (rows) × condition (columns)
ct = pd.crosstab(df['protein'], df['condition'])
ct['TOTAL'] = ct.sum(axis=1)
ct.loc['TOTAL'] = ct.sum()
print('Sample counts per group:')
print(ct.to_string())
print()
print('Notes:')
print('  • RhoA + TNF: n=1  → too few for statistical testing, descriptive only')
print('  • RhoB + TNF: n=4  → sufficient for Mann-Whitney U test (SQ3)')
print('  • Quiescent:  n=18 → well-sampled control group')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: samples per condition × protein ────────────────────────────────
ax = axes[0]
groups = df.groupby(['protein', 'condition']).size().unstack(fill_value=0)
x = np.arange(len(groups.index))  # RhoA, RhoB
w = 0.35
conditions = groups.columns.tolist()
colours = {'Quiescent': 'steelblue', 'TNF': 'tomato'}
for i, cond in enumerate(conditions):
    ax.bar(x + i * w, groups[cond], w, label=cond,
           color=colours.get(cond, 'grey'), edgecolor='white', linewidth=0.5)
    for xi, v in zip(x + i * w, groups[cond]):
        ax.text(xi, v + 0.1, str(v), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xticks(x + w / 2)
ax.set_xticklabels(groups.index)
ax.set_xlabel('Rho GTPase protein')
ax.set_ylabel('Number of samples')
ax.set_title('Samples per Condition × Protein')
ax.legend(title='Condition')
ax.set_ylim(0, 14)

# ── Right: total frames per condition ────────────────────────────────────
ax2 = axes[1]
frames_by_cond = df.groupby('condition')['n_frames'].sum()
bars = ax2.bar(frames_by_cond.index,
               frames_by_cond.values,
               color=[colours.get(c, 'grey') for c in frames_by_cond.index],
               edgecolor='white')
for bar, v in zip(bars, frames_by_cond.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 5, str(v),
             ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2.set_xlabel('Condition')
ax2.set_ylabel('Total frames')
ax2.set_title('Total Time Frames per Condition')
ax2.set_ylim(0, 1200)

plt.tight_layout()
plt.savefig(EDA_DIR / 'dataset_condition_protein_breakdown.png', bbox_inches='tight')
plt.show()
print(f'Saved → {EDA_DIR / "dataset_condition_protein_breakdown.png"}')

## 4. Frames Per Sample — how long is each recording?

Each sample is a time-lapse recording. More frames = a longer recording = more
time points to track cell behaviour over.

Tracking requires linking the same cell across consecutive frames.
Samples with very few frames may not be suitable for trajectory analysis.

> Frame interval in the experiment: **~5 minutes per frame**.  
> So 46 frames ≈ ~3.8 hours of recording.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: bar chart, one bar per sample ──────────────────────────────────
ax = axes[0]
df_sorted = df.sort_values('n_frames', ascending=False).reset_index(drop=True)
bar_colours = ['tomato' if c == 'TNF' else 'steelblue' for c in df_sorted['condition']]
ax.bar(range(len(df_sorted)), df_sorted['n_frames'], color=bar_colours, edgecolor='white')
ax.axhline(df['n_frames'].median(), color='black', linestyle='--', linewidth=1.2,
           label=f'Median = {df["n_frames"].median():.0f}')
ax.set_xlabel('Sample (sorted by frame count)')
ax.set_ylabel('Number of frames')
ax.set_title('Frames per Sample')
ax.legend()
patch_q   = mpatches.Patch(color='steelblue', label='Quiescent')
patch_tnf = mpatches.Patch(color='tomato',    label='TNF')
ax.legend(handles=[patch_q, patch_tnf,
                   plt.Line2D([0],[0], color='black', linestyle='--', label='Median')])

# ── Right: histogram of frame counts ─────────────────────────────────────
ax2 = axes[1]
ax2.hist(df[df['condition'] == 'Quiescent']['n_frames'], bins=10,
         alpha=0.7, color='steelblue', label='Quiescent', edgecolor='white')
ax2.hist(df[df['condition'] == 'TNF']['n_frames'], bins=6,
         alpha=0.7, color='tomato', label='TNF', edgecolor='white')
ax2.set_xlabel('Number of frames')
ax2.set_ylabel('Count of samples')
ax2.set_title('Frame Count Distribution')
ax2.legend()

plt.suptitle('Time-lapse Recording Lengths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(EDA_DIR / 'dataset_frame_counts.png', bbox_inches='tight')
plt.show()
print(f'Saved → {EDA_DIR / "dataset_frame_counts.png"}')

# Print descriptive stats
print('\nFrame count statistics:')
print(df.groupby('condition')['n_frames'].describe().round(1).to_string())

## 5. Complete Sample List

Here is every sample we have, with its key metadata.

**How to read the table:**
- `folder` = which experiment batch this sample came from (by date)
- `protein` = which Rho GTPase was fluorescently tagged (RhoA or RhoB)
- `condition` = Quiescent (resting) or TNF (inflamed)
- `n_frames` = how many time points in this recording
- `n_cells` = how many cells Cellpose detected in frame 1
- `coverage` = what fraction of the image area is covered by detected cells

In [ ]:
display_cols = ['sample_name', 'folder', 'protein', 'condition',
                'n_frames', 'n_cells', 'coverage']

display_df = (
    df[display_cols]
    .sort_values(['protein', 'condition', 'sample_name'])
    .reset_index(drop=True)
)
display_df['coverage'] = (display_df['coverage'] * 100).round(1).astype(str) + '%'

# Shorten sample_name for readability
display_df['sample_name'] = display_df['sample_name'].str.replace(
    'HUVECp4_veGFP_mCHA_siractin_', '', regex=False
).str.replace(
    'HUVECp4_veGFP_mCHB_siractin_', '', regex=False
).str.replace(
    'HUVECp5_veGFP_mChA_siractin_', '', regex=False
).str.replace(
    'HUVECp5_veGFP_mChB_siractin_', '', regex=False
)

print(display_df.to_string(index=False))

## 6. Experiment Dates — batch effects?

The data was collected across **6 experiment sessions** on different dates.
In biology, samples from different dates can behave slightly differently
even under the same conditions — this is called a **batch effect**.

We check: how are samples distributed across experiment dates?
If all TNF samples happen to be from one date and all Q from another,
any difference we see might be a date effect, not a biology effect.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# Group by folder, condition
folder_cond = df.groupby(['folder', 'condition']).size().unstack(fill_value=0)
folder_cond = folder_cond.reindex(columns=['Quiescent', 'TNF'], fill_value=0)

x = np.arange(len(folder_cond))
w = 0.35
ax.bar(x - w/2, folder_cond['Quiescent'], w, color='steelblue',
       label='Quiescent', edgecolor='white')
ax.bar(x + w/2, folder_cond['TNF'], w, color='tomato',
       label='TNF', edgecolor='white')

ax.set_xticks(x)
# Shorten folder names
short_labels = [f.replace('JK2513_', '').replace('JK2513_CPM_', '') for f in folder_cond.index]
ax.set_xticklabels(short_labels, rotation=30, ha='right')
ax.set_ylabel('Number of samples')
ax.set_title('Samples per Experiment Date (Batch)')
ax.legend(title='Condition')
ax.set_ylim(0, 7)

plt.tight_layout()
plt.savefig(EDA_DIR / 'dataset_experiment_batches.png', bbox_inches='tight')
plt.show()

print('\nSamples per experiment folder:')
print(folder_cond.to_string())
print('\nObservation: TNF samples are spread across 3 different experiment dates,')
print('reducing the risk of a pure batch effect.')

## 7. Dataset Summary Figure (thesis-ready)

A clean single figure combining the most important dataset statistics,
suitable for the *Data Description* section of the methodology chapter.

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = fig.add_gridspec(1, 3, wspace=0.35)

# Panel A: sample count by condition
ax1 = fig.add_subplot(gs[0])
cond_counts = df['condition'].value_counts()
ax1.pie(cond_counts.values,
        labels=cond_counts.index,
        colors=['steelblue', 'tomato'],
        autopct='%1.0f%%',
        startangle=90,
        textprops={'fontsize': 11})
ax1.set_title('A  Samples by condition', fontweight='bold', pad=10)

# Panel B: sample count by protein
ax2 = fig.add_subplot(gs[1])
prot_counts = df['protein'].value_counts()
ax2.pie(prot_counts.values,
        labels=prot_counts.index,
        colors=['#5B9BD5', '#ED7D31'],
        autopct='%1.0f%%',
        startangle=90,
        textprops={'fontsize': 11})
ax2.set_title('B  Samples by protein', fontweight='bold', pad=10)

# Panel C: frames per sample (box + strip)
ax3 = fig.add_subplot(gs[2])
q_frames   = df[df['condition'] == 'Quiescent']['n_frames']
tnf_frames = df[df['condition'] == 'TNF']['n_frames']
bp = ax3.boxplot([q_frames, tnf_frames],
                 labels=['Quiescent', 'TNF'],
                 patch_artist=True,
                 medianprops={'color': 'black', 'linewidth': 2})
for patch, colour in zip(bp['boxes'], ['steelblue', 'tomato']):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)
# Overlay individual points
ax3.scatter([1]*len(q_frames),   q_frames,   color='steelblue', alpha=0.6, zorder=3)
ax3.scatter([2]*len(tnf_frames), tnf_frames, color='tomato',    alpha=0.6, zorder=3)
ax3.set_ylabel('Number of frames per sample')
ax3.set_title('C  Recording length', fontweight='bold')

fig.suptitle(
    f'Dataset Overview: {len(df)} HUVEC samples, '
    f'{df["n_frames"].sum()} total frames',
    fontsize=13, fontweight='bold'
)
plt.savefig(EDA_DIR / 'dataset_summary_figure.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved → {EDA_DIR / "dataset_summary_figure.png"}')

## Key Takeaways

| Finding | Implication for thesis |
|---------|------------------------|
| **23 samples total** (18 Q, 5 TNF) | Dataset is imbalanced — more Quiescent samples than TNF |
| **RhoA + TNF: n=1** | Cannot run statistical tests for this group; descriptive only |
| **RhoB + TNF: n=4** | Sufficient for Mann-Whitney U test against RhoB Quiescent (n=8) |
| **Frames: 19–95 per sample** | High variability; shorter recordings limit tracking analysis |
| **TNF samples spread across 3 dates** | Reduces risk of batch effect confounding Q vs TNF comparison |
| **6 experiment batches** | Should report batch information; consider as covariate if needed |

### Decisions for the thesis:
1. **Primary comparison**: RhoB Quiescent (n=8) vs RhoB TNF (n=4) — statistically testable
2. **Secondary comparison**: RhoA Quiescent (n=10) vs all Q — within-control variation
3. **RhoA TNF (n=1)**: Included for completeness but not statistically compared
4. **Temporal analysis**: Only samples with ≥30 frames will be used for tracking-based features

---
*Next: [EDA Part 2 — Raw Image Analysis](eda_02_raw_images.ipynb)*